In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install timm scikit-learn

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as transforms
import timm

from sklearn.metrics import accuracy_score, mean_absolute_error

In [ ]:
class RetinaDataset(Dataset):

    def __init__(self, csv_file, image_dir, transform=None):

        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        img_name = row["image_name"]
        thickness = float(row["thickness"])
        label = int(row["label"])
        age = float(row["True_age"])

        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, thickness, label, age

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [ ]:
csv_file = "/kaggle/input/yourdataset/labels.csv"
image_dir = "/kaggle/input/yourdataset/images"

dataset = RetinaDataset(csv_file, image_dir, transform)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset,[train_size,val_size])

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True,num_workers=0)
val_loader = DataLoader(val_dataset,batch_size=16,num_workers=0)

In [ ]:
class RetinaModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = timm.create_model(
            "convnext_base",
            pretrained=True,
            num_classes=0
        )

        self.thickness_head = nn.Linear(1024,1)
        self.label_head = nn.Linear(1024,2)
        self.age_head = nn.Linear(1024,1)

    def forward(self,x):

        f = self.backbone(x)

        thickness = self.thickness_head(f)
        label = self.label_head(f)
        age = self.age_head(f)

        return thickness,label,age

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = RetinaModel().to(device)

In [ ]:
regression_loss = nn.MSELoss()
classification_loss = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for imgs, thickness, label, age in train_loader:

        imgs = imgs.to(device)

        thickness = thickness.float().unsqueeze(1).to(device)
        label = label.long().to(device)
        age = age.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        pred_thickness, pred_label, pred_age = model(imgs)

        loss1 = regression_loss(pred_thickness, thickness)
        loss2 = classification_loss(pred_label, label)
        loss3 = regression_loss(pred_age, age)

        loss = loss1 + loss2 + loss3

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    print("Epoch:",epoch+1,"Train Loss:",train_loss)

In [ ]:
torch.save(model.state_dict(),"retina_model.pth")

In [ ]:
model = RetinaModel().to(device)

model.load_state_dict(torch.load("retina_model.pth",map_location=device))

model.eval()

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
def predict_image(image_path):

    img = Image.open(image_path).convert("RGB")

    img = test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():

        thickness,label,age = model(img)

    thickness = thickness.item()
    label = torch.argmax(label).item()
    age = age.item()

    print("Predicted Thickness:", thickness)
    print("Predicted Label:", label)
    print("Predicted Age:", age)

    return thickness

In [ ]:
# predict_image("/kaggle/input/testdataset/test_image.png")